In [2]:
import polars as pl
from pathlib import Path
from glob import glob

DATA = Path("../data")
OUTPUTS = Path("../outputs")

labels   = pl.read_parquet(DATA / "train_labels.parquet")
accounts = pl.read_parquet(DATA / "accounts.parquet")
train    = pl.read_parquet(OUTPUTS / "train_model_ready.parquet")

print(f"Labels: {labels.shape}")
print(f"Train ready: {train.shape}")

Labels: (96091, 5)
Train ready: (96091, 71)


In [3]:
# Step 1 — get global median and std per MCC code from a sample first
# We use all batches but only keep mcc_code and amount

def compute_mcc_stats(batch_parts):
    return (
        pl.scan_parquet(batch_parts)
        .select(["account_id", "mcc_code", "amount"])
        .filter(pl.col("amount") > 0)  # credits only
        .collect()
    )

print("Computing MCC statistics per batch...")
mcc_results = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    parts = sorted(glob(f"../data/transactions/{batch}/part_*.parquet"))
    print(f"Processing {batch}...")
    result = compute_mcc_stats(parts)
    mcc_results.append(result)
    print(f"  Done: {result.shape}")

mcc_all = pl.concat(mcc_results)
print(f"\nTotal rows: {len(mcc_all)}")
print(f"Unique MCC codes: {mcc_all['mcc_code'].n_unique()}")
print(f"\nTop 15 most common MCC codes:")
print(mcc_all["mcc_code"].value_counts().sort("count", descending=True).head(15))

Computing MCC statistics per batch...
Processing batch-1...
  Done: (100189938, 3)
Processing batch-2...
  Done: (99912565, 3)
Processing batch-3...
  Done: (99964067, 3)
Processing batch-4...
  Done: (95848526, 3)

Total rows: 395915096
Unique MCC codes: 248

Top 15 most common MCC codes:
shape: (15, 2)
┌──────────┬──────────┐
│ mcc_code ┆ count    │
│ ---      ┆ ---      │
│ i64      ┆ u32      │
╞══════════╪══════════╡
│ 9384     ┆ 83593536 │
│ 5682     ┆ 68108645 │
│ 9355     ┆ 52954665 │
│ 5651     ┆ 52948879 │
│ 5905     ┆ 41391859 │
│ …        ┆ …        │
│ 2769     ┆ 4940698  │
│ 0        ┆ 4852708  │
│ 1139     ┆ 3790382  │
│ 4702     ┆ 3036820  │
│ 2920     ┆ 3035993  │
└──────────┴──────────┘


In [4]:
# Global stats per MCC
global_mcc_stats = (
    mcc_all
    .group_by("mcc_code")
    .agg([
        pl.col("amount").median().alias("global_mcc_median"),
        pl.col("amount").std().alias("global_mcc_std"),
        pl.col("amount").mean().alias("global_mcc_mean"),
        pl.len().alias("global_mcc_txn_count"),
    ])
)

print(f"MCC benchmark table shape: {global_mcc_stats.shape}")
print(global_mcc_stats.sort("global_mcc_txn_count", descending=True).head(10))

MCC benchmark table shape: (248, 5)
shape: (10, 5)
┌──────────┬───────────────────┬────────────────┬─────────────────┬──────────────────────┐
│ mcc_code ┆ global_mcc_median ┆ global_mcc_std ┆ global_mcc_mean ┆ global_mcc_txn_count │
│ ---      ┆ ---               ┆ ---            ┆ ---             ┆ ---                  │
│ i64      ┆ f64               ┆ f64            ┆ f64             ┆ u32                  │
╞══════════╪═══════════════════╪════════════════╪═════════════════╪══════════════════════╡
│ 9384     ┆ 414.61            ┆ 7147.728438    ┆ 1726.744904     ┆ 83593536             │
│ 5682     ┆ 163.31            ┆ 22110.986929   ┆ 979.200183      ┆ 68108645             │
│ 9355     ┆ 1000.0            ┆ 159575.641919  ┆ 12328.374654    ┆ 52954665             │
│ 5651     ┆ 1000.0            ┆ 133318.303232  ┆ 11027.972873    ┆ 52948879             │
│ 5905     ┆ 6300.0            ┆ 689161.492654  ┆ 69438.945506    ┆ 41391859             │
│ 2557     ┆ 6568.2            ┆ 873396

In [5]:
global_mcc_stats = pl.read_parquet(OUTPUTS / "global_mcc_stats.parquet") \
    if (OUTPUTS / "global_mcc_stats.parquet").exists() \
    else global_mcc_stats

# Save global stats first so we don't lose it
global_mcc_stats.write_parquet(OUTPUTS / "global_mcc_stats.parquet")

print("Computing per-account MCC features batch by batch...")

def compute_account_mcc_batch(batch_parts, global_stats):
    return (
        pl.scan_parquet(batch_parts)
        .select(["account_id", "mcc_code", "amount"])
        .filter(pl.col("amount") > 0)
        .group_by(["account_id", "mcc_code"])
        .agg([
            pl.col("amount").mean().alias("acct_mcc_avg"),
            pl.len().alias("acct_mcc_txn_count"),
        ])
        .collect()
        .join(global_stats, on="mcc_code", how="left")
        .with_columns([
            ((pl.col("acct_mcc_avg") - pl.col("global_mcc_mean")) /
             (pl.col("global_mcc_std") + 1)).alias("mcc_amount_zscore"),
        ])
        .filter(pl.col("acct_mcc_txn_count") >= 3)
        .group_by("account_id")
        .agg([
            pl.col("mcc_amount_zscore").max().alias("max_mcc_zscore"),
            pl.col("mcc_amount_zscore").mean().alias("avg_mcc_zscore"),
            pl.col("mcc_amount_zscore").abs().max().alias("max_abs_mcc_zscore"),
            pl.col("mcc_code").n_unique().alias("unique_mcc_codes"),
            (pl.col("mcc_amount_zscore").abs() > 2).sum().alias("anomalous_mcc_count"),
        ])
    )

batch_mcc = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    parts = sorted(glob(f"../data/transactions/{batch}/part_*.parquet"))
    print(f"Processing {batch}...")
    result = compute_account_mcc_batch(parts, global_mcc_stats)
    batch_mcc.append(result)
    print(f"  Done: {result.shape}")

# Combine — same account appears in multiple batches, re-aggregate
print("\nCombining batches...")
mcc_features = (
    pl.concat(batch_mcc)
    .group_by("account_id")
    .agg([
        pl.col("max_mcc_zscore").max(),
        pl.col("avg_mcc_zscore").mean(),
        pl.col("max_abs_mcc_zscore").max(),
        pl.col("unique_mcc_codes").max(),
        pl.col("anomalous_mcc_count").max(),
    ])
)

mcc_features.write_parquet(OUTPUTS / "features_mcc.parquet")
print(f"Saved. Shape: {mcc_features.shape}")

# Signal check
check = labels.join(mcc_features, on="account_id", how="left")
mule_c = check.filter(pl.col("is_mule") == 1)
legit_c = check.filter(pl.col("is_mule") == 0)

print(f"\n{'Feature':<25} {'Mule':>10} {'Legit':>10} {'Ratio':>8}")
print("-" * 57)
for feat in ["max_mcc_zscore", "avg_mcc_zscore",
             "max_abs_mcc_zscore", "unique_mcc_codes", "anomalous_mcc_count"]:
    m = mule_c[feat].median()
    l = legit_c[feat].median()
    ratio = m/l if l and abs(l) > 0.001 else float('inf')
    print(f"{feat:<25} {m:>10.3f} {l:>10.3f} {ratio:>8.2f}x")

Computing per-account MCC features batch by batch...
Processing batch-1...
  Done: (40069, 6)
Processing batch-2...
  Done: (39655, 6)
Processing batch-3...
  Done: (39441, 6)
Processing batch-4...
  Done: (47375, 6)

Combining batches...
Saved. Shape: (159401, 6)

Feature                         Mule      Legit    Ratio
---------------------------------------------------------
max_mcc_zscore                 1.080      0.595     1.82x
avg_mcc_zscore                 0.024     -0.016    -1.50x
max_abs_mcc_zscore             1.080      0.595     1.82x
unique_mcc_codes              37.000     37.000     1.00x
anomalous_mcc_count            0.000      0.000      infx


In [6]:
import polars as pl
from pathlib import Path
from glob import glob

DATA = Path("../data")
OUTPUTS = Path("../outputs")
labels = pl.read_parquet(DATA / "train_labels.parquet")

def process_ip_balance_batch(batch_name):
    txn_parts = sorted(glob(f"../data/transactions/{batch_name}/part_*.parquet"))
    txn_add_parts = sorted(glob(
        f"../data/transactions_additional/{batch_name}/part_*.parquet"))
    
    print(f"Loading {batch_name}: {len(txn_parts)} txn + {len(txn_add_parts)} add parts...")
    
    txn_slim = (
        pl.scan_parquet(txn_parts)
        .select(["transaction_id", "account_id"])
        .collect()
    )
    print(f"  txn_slim loaded: {txn_slim.shape}")
    
    txn_add = (
        pl.scan_parquet(txn_add_parts)
        .select(["transaction_id", "ip_address",
                 "balance_after_transaction"])
        .collect()
    )
    print(f"  txn_add loaded: {txn_add.shape}")
    
    joined = (
        txn_add
        .join(txn_slim, on="transaction_id", how="left")
        .filter(pl.col("account_id").is_not_null())
    )
    print(f"  joined: {joined.shape}")
    return joined

batch1 = process_ip_balance_batch("batch-1")
batch1.write_parquet(OUTPUTS / "temp_ip_balance_batch1.parquet")
print("batch-1 saved.")
del batch1

Loading batch-1: 100 txn + 100 add parts...
  txn_slim loaded: (100686033, 2)
  txn_add loaded: (129898887, 3)
  joined: (100686033, 4)
batch-1 saved.


In [7]:
batch2 = process_ip_balance_batch("batch-2")
batch2.write_parquet(OUTPUTS / "temp_ip_balance_batch2.parquet")
print("batch-2 saved.")
del batch2

Loading batch-2: 100 txn + 100 add parts...
  txn_slim loaded: (100407662, 2)
  txn_add loaded: (126415150, 3)
  joined: (71194808, 4)
batch-2 saved.


In [8]:
batch3 = process_ip_balance_batch("batch-3")
batch3.write_parquet(OUTPUTS / "temp_ip_balance_batch3.parquet")
print("batch-3 saved.")
del batch3

Loading batch-3: 100 txn + 100 add parts...
  txn_slim loaded: (100460390, 2)
  txn_add loaded: (128875774, 3)
  joined: (45240048, 4)
batch-3 saved.


In [9]:
batch4 = process_ip_balance_batch("batch-4")
batch4.write_parquet(OUTPUTS / "temp_ip_balance_batch4.parquet")
print("batch-4 saved.")
del batch4

Loading batch-4: 96 txn + 11 add parts...
  txn_slim loaded: (96321238, 2)
  txn_add loaded: (12685512, 3)
  joined: (12685512, 4)
batch-4 saved.


In [10]:
print("Loading all batches...")
all_data = pl.concat([
    pl.read_parquet(OUTPUTS / "temp_ip_balance_batch1.parquet"),
    pl.read_parquet(OUTPUTS / "temp_ip_balance_batch2.parquet"),
    pl.read_parquet(OUTPUTS / "temp_ip_balance_batch3.parquet"),
    pl.read_parquet(OUTPUTS / "temp_ip_balance_batch4.parquet"),
])
print(f"Combined: {all_data.shape}")

# IP sharing — how many accounts share same IP
ip_sharing = (
    all_data
    .filter(pl.col("ip_address").is_not_null())
    .select(["account_id", "ip_address"])
    .unique()
    .group_by("ip_address")
    .agg(pl.col("account_id").n_unique().alias("accounts_per_ip"))
)

# Per account IP features
ip_features = (
    all_data
    .filter(pl.col("ip_address").is_not_null())
    .select(["account_id", "ip_address"])
    .unique()
    .join(ip_sharing, on="ip_address", how="left")
    .group_by("account_id")
    .agg([
        pl.col("ip_address").n_unique().alias("unique_ip_count"),
        pl.col("accounts_per_ip").max().alias("max_accounts_sharing_ip"),
        pl.col("accounts_per_ip").mean().alias("avg_accounts_sharing_ip"),
    ])
)

# Balance features
balance_features = (
    all_data
    .filter(pl.col("balance_after_transaction").is_not_null())
    .group_by("account_id")
    .agg([
        pl.col("balance_after_transaction").min().alias("min_balance"),
        pl.col("balance_after_transaction").max().alias("max_balance"),
        pl.col("balance_after_transaction").mean().alias("mean_balance"),
        pl.col("balance_after_transaction").std().alias("std_balance"),
        (pl.col("balance_after_transaction") < 100).sum().alias("near_zero_count"),
        (pl.col("balance_after_transaction") < 0).sum().alias("negative_balance_count"),
    ])
    .with_columns([
        (pl.col("min_balance") / (pl.col("max_balance").abs() + 1))
        .alias("balance_drain_ratio"),
        (pl.col("std_balance") / (pl.col("mean_balance").abs() + 1))
        .alias("balance_volatility"),
    ])
)

# Combine
ip_balance_features = ip_features.join(balance_features, on="account_id", how="outer")
ip_balance_features.write_parquet(OUTPUTS / "features_ip_balance.parquet")
print(f"Saved. Shape: {ip_balance_features.shape}")

# Signal check
check = labels.join(ip_balance_features, on="account_id", how="left")
mule_c = check.filter(pl.col("is_mule") == 1)
legit_c = check.filter(pl.col("is_mule") == 0)

print(f"\n{'Feature':<30} {'Mule':>10} {'Legit':>10} {'Ratio':>8}")
print("-" * 62)
for feat in ["unique_ip_count", "max_accounts_sharing_ip",
             "avg_accounts_sharing_ip", "min_balance",
             "near_zero_count", "balance_drain_ratio",
             "balance_volatility", "negative_balance_count"]:
    try:
        m = mule_c[feat].median()
        l = legit_c[feat].median()
        ratio = m/l if l and abs(l) > 0.001 else float('inf')
        print(f"{feat:<30} {m:>10.3f} {l:>10.3f} {ratio:>8.2f}x")
    except Exception as e:
        print(f"{feat:<30} ERROR: {e}")

Loading all batches...
Combined: (229806401, 4)


/var/folders/dt/96p55h2n6119wn13mj_0v5fh0000gn/T/ipykernel_67294/2083407457.py:57: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  ip_balance_features = ip_features.join(balance_features, on="account_id", how="outer")


Saved. Shape: (98692, 13)

Feature                              Mule      Legit    Ratio
--------------------------------------------------------------
unique_ip_count                   532.000    660.000     0.81x
max_accounts_sharing_ip             6.000      6.000     1.00x
avg_accounts_sharing_ip             1.942      1.944     1.00x
min_balance                    -4900882.200 -6579773.330     0.74x
near_zero_count                   819.500   1093.000     0.75x
balance_drain_ratio                -2.471     -2.637     0.94x
balance_volatility                  0.898      0.879     1.02x
negative_balance_count            809.000   1090.000     0.74x


In [11]:
import os

# Delete temp batch files — no longer needed
for i in range(1, 5):
    f = OUTPUTS / f"temp_ip_balance_batch{i}.parquet"
    if f.exists():
        os.remove(f)
        print(f"Deleted {f}")

# Delete the dead feature file
dead = OUTPUTS / "features_ip_balance.parquet"
if dead.exists():
    os.remove(dead)
    print(f"Deleted {dead}")

print("Cleanup done.")

Deleted ../outputs/temp_ip_balance_batch1.parquet
Deleted ../outputs/temp_ip_balance_batch2.parquet
Deleted ../outputs/temp_ip_balance_batch3.parquet
Deleted ../outputs/temp_ip_balance_batch4.parquet
Deleted ../outputs/features_ip_balance.parquet
Cleanup done.


In [12]:
train_model = pl.read_parquet(OUTPUTS / "train_model_ready.parquet")
mcc_features = pl.read_parquet(OUTPUTS / "features_mcc.parquet")

train_final = (
    train_model
    .join(mcc_features.select([
        "account_id", "max_mcc_zscore", 
        "avg_mcc_zscore", "max_abs_mcc_zscore"
    ]), on="account_id", how="left")
    .with_columns([
        pl.col("max_mcc_zscore").fill_null(0),
        pl.col("avg_mcc_zscore").fill_null(0),
        pl.col("max_abs_mcc_zscore").fill_null(0),
    ])
)

train_final.write_parquet(OUTPUTS / "train_model_ready.parquet")
print(f"Final training file: {train_final.shape}")
print(f"Mules: {train_final['is_mule'].sum()}")

ColumnNotFoundError: unable to find column "account_id"; valid columns: ["is_mule", "total_txn_count", "avg_amount", "std_amount", "total_amount", "max_amount", "min_amount", "reversal_count", "credit_count", "debit_count", "unique_channels", "unique_counterparties", "atm_count", "upi_credit_count", "upi_debit_count", "imps_count", "neft_count", "standing_instr_count", "avg_txn_hour", "late_night_txn_count", "credit_ratio", "debit_ratio", "atm_rate", "upi_rate", "interbank_rate", "standing_instr_rate", "reversal_rate", "amount_volatility", "active_span_days", "txn_velocity", "late_night_rate", "passthrough_rate", "active_days", "burst_ratio", "std_monthly_txn", "active_months", "counterparty_entropy", "unique_cp_count", "pct_volume_top3", "mule_network_cp_count", "mule_cp_weighted_score", "contamination_rate", "max_mule_cp_connection", "account_age_days", "is_frozen", "days_since_mobile_update", "had_mobile_update", "balance_monthly_quarterly_diff", "balance_daily_monthly_diff", "freez

In [13]:
# Load master which still has account_id
master = pl.read_parquet(OUTPUTS / "master_features_all.parquet")
mcc_features = pl.read_parquet(OUTPUTS / "features_mcc.parquet")

# Check what account_id looks like in master
print(master.select(["account_id"]).head(3))
print(f"MCC features columns: {mcc_features.columns}")

shape: (3, 1)
┌─────────────┐
│ account_id  │
│ ---         │
│ str         │
╞═════════════╡
│ ACCT_000000 │
│ ACCT_000002 │
│ ACCT_000003 │
└─────────────┘
MCC features columns: ['account_id', 'max_mcc_zscore', 'avg_mcc_zscore', 'max_abs_mcc_zscore', 'unique_mcc_codes', 'anomalous_mcc_count']


In [14]:
labels = pl.read_parquet(DATA / "train_labels.parquet")
contamination = pl.read_parquet(OUTPUTS / "features_contamination.parquet")
entropy = pl.read_parquet(OUTPUTS / "features_entropy.parquet")
passthrough = pl.read_parquet(OUTPUTS / "features_passthrough.parquet")
burst = pl.read_parquet(OUTPUTS / "features_burst.parquet")
txn_derived = pl.read_parquet(OUTPUTS / "features_txn_derived.parquet")
account_feat = pl.read_parquet(OUTPUTS / "features_account.parquet")
branch_feat = pl.read_parquet(OUTPUTS / "features_branch.parquet")
customer_feat = pl.read_parquet(OUTPUTS / "features_customer.parquet")
accounts = pl.read_parquet(DATA / "accounts.parquet")
linkage = pl.read_parquet(DATA / "customer_account_linkage.parquet")

EXCLUDE = [
    "alert_reason", "mule_flag_date", "flagged_by_branch",
    "first_txn_date", "last_txn_date",
    "branch_code", "customer_id",
    "product_family", "rural_branch", "branch_type_encoded",
    "total_amount_right", "avg_amount_right",
]

train_final = (
    labels
    .join(txn_derived, on="account_id", how="left")
    .join(passthrough.select(["account_id", "passthrough_rate", "active_days"]),
          on="account_id", how="left")
    .join(burst.select(["account_id", "burst_ratio", "std_monthly_txn", "active_months"]),
          on="account_id", how="left")
    .join(entropy.select(["account_id", "counterparty_entropy", "unique_cp_count", "pct_volume_top3"]),
          on="account_id", how="left")
    .join(contamination.select(["account_id", "mule_network_cp_count",
                                 "mule_cp_weighted_score", "contamination_rate",
                                 "max_mule_cp_connection"]),
          on="account_id", how="left")
    .join(account_feat.drop(["branch_code"]), on="account_id", how="left")
    .join(
        accounts.select(["account_id", "branch_code"]),
        on="account_id", how="left"
    )
    .join(branch_feat.select(["branch_code", "branch_mule_rate",
                               "branch_relative_risk", "mules_per_employee",
                               "branch_turnover", "branch_asset_size"]),
          on="branch_code", how="left")
    .join(linkage, on="account_id", how="left")
    .join(customer_feat, on="customer_id", how="left")
    .join(mcc_features.select(["account_id", "max_mcc_zscore",
                                "avg_mcc_zscore", "max_abs_mcc_zscore"]),
          on="account_id", how="left")
    .drop([c for c in EXCLUDE if c in labels.columns or 
           c in txn_derived.columns or c in account_feat.columns])
    .with_columns([
        pl.col("days_since_mobile_update").fill_null(9999),
        pl.col("days_since_passbook_update").fill_null(9999),
        pl.col("days_since_address_update").fill_null(9999),
        pl.col("freeze_duration_days").fill_null(0),
        pl.col("kyc_doc_score").fill_null(0),
        pl.col("active_span_days").fill_null(0),
        pl.col("txn_velocity").fill_null(0),
        pl.col("avg_txn_hour").fill_null(12),
        pl.col("std_monthly_txn").fill_null(0),
        pl.col("burst_ratio").fill_null(1),
        pl.col("avg_balance").fill_null(pl.col("avg_balance").median()),
        pl.col("monthly_avg_balance").fill_null(pl.col("monthly_avg_balance").median()),
        pl.col("balance_monthly_quarterly_diff").fill_null(0),
        pl.col("balance_daily_monthly_diff").fill_null(0),
        pl.col("balance_to_txn_ratio").fill_null(0),
        pl.col("volume_to_balance_ratio").fill_null(0),
        pl.col("max_mcc_zscore").fill_null(0),
        pl.col("avg_mcc_zscore").fill_null(0),
        pl.col("max_abs_mcc_zscore").fill_null(0),
    ])
)

# Verify no nulls in model features
model_cols = [c for c in train_final.columns
              if c not in ["is_mule", "account_id", "mule_flag_date",
                           "alert_reason", "flagged_by_branch"]]
null_check = train_final.select(model_cols).null_count()
remaining = [(c, null_check[c][0]) for c in null_check.columns 
             if null_check[c][0] > 0]

if remaining:
    print("Remaining nulls:")
    for c, n in sorted(remaining, key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {c}: {n}")
else:
    print("No nulls remaining.")

train_final.write_parquet(OUTPUTS / "train_model_ready.parquet")
print(f"\nFinal shape: {train_final.shape}")
print(f"Columns: {train_final.columns}")

No nulls remaining.

Final shape: (96091, 78)
Columns: ['account_id', 'is_mule', 'total_txn_count', 'avg_amount', 'std_amount', 'total_amount', 'max_amount', 'min_amount', 'reversal_count', 'credit_count', 'debit_count', 'unique_channels', 'unique_counterparties', 'atm_count', 'upi_credit_count', 'upi_debit_count', 'imps_count', 'neft_count', 'standing_instr_count', 'avg_txn_hour', 'late_night_txn_count', 'credit_ratio', 'debit_ratio', 'atm_rate', 'upi_rate', 'interbank_rate', 'standing_instr_rate', 'reversal_rate', 'amount_volatility', 'active_span_days', 'txn_velocity', 'late_night_rate', 'passthrough_rate', 'active_days', 'burst_ratio', 'std_monthly_txn', 'active_months', 'counterparty_entropy', 'unique_cp_count', 'pct_volume_top3', 'mule_network_cp_count', 'mule_cp_weighted_score', 'contamination_rate', 'max_mule_cp_connection', 'account_age_days', 'is_frozen', 'days_since_mobile_update', 'had_mobile_update', 'balance_monthly_quarterly_diff', 'balance_daily_monthly_diff', 'freeze_d

In [15]:
# Drop the leftover duplicate and identifier columns
train_final = train_final.drop(["total_amount_right", "avg_amount_right", "customer_id"])
train_final.write_parquet(OUTPUTS / "train_model_ready.parquet")
print(f"Final shape: {train_final.shape}")
print(f"Mules: {train_final['is_mule'].sum()}")

Final shape: (96091, 75)
Mules: 2683
